# SIFLOW · Notebook 2 · Dream-7B & LLaDA-8B (Table 2 -D / -L) + final artifacts

Head-only SIFLOW on two larger backbones, each fitting a single A100-40GB in fp16 (no quantization): **Dream-7B** (~14 GB, -D) then **LLaDA-8B** (~16 GB, -L), trained and evaluated sequentially with the teacher freed between them. Then regenerates the FINAL combined tables + figures from every results JSON (the imported MDLM rows plus -D/-L). Downloads `nb2_final_paper_artifacts.zip` for the paper.

**Runtime:** designed to finish in one Colab session on a single **A100-40GB**. Every stage is checkpointed and guarded by an existence check, and training has an 11h wall-clock guard — if a session ends early, just re-run this notebook (re-upload its own zip) and it skips finished stages and resumes training.

**How to use:** run every cell top-to-bottom. Cells 1–2 set up the environment and the artifact location; the last cell downloads this part's output zip.

In [ ]:
# === 1. Clone + install (run once per session, ~2 min) ===
REPO_URL = "https://github.com/kagtgi/siflow.git"
import os
if not os.path.isdir("siflow"):
    !git clone $REPO_URL siflow
%cd siflow
!git pull -q
!pip -q install -e .
!pip -q install -r requirements-colab.txt
print("setup done")

In [ ]:
# === 2. Where do artifacts live? ===
# Default: a local folder + zip download/upload between the 2 notebooks (no Drive).
# Set USE_DRIVE = True to persist on Google Drive instead (the import + download
# steps then become no-ops and everything survives a disconnect automatically).
USE_DRIVE = False

import os
if USE_DRIVE:
    from siflow.utils import drive
    drive.mount()
    os.environ["SIFLOW_BASE"] = "/content/drive/MyDrive/siflow"
    BASE = drive.base_dir()
else:
    BASE = "/content/artifacts"
    os.makedirs(BASE, exist_ok=True)
print("artifacts ->", BASE)

In [ ]:
# === Hugging Face login ===
# Recommended for the Dream-7B / LLaDA-8B weights (faster, avoids rate limits).
# Get a token at https://huggingface.co/settings/tokens (read scope).
from huggingface_hub import login
login()

### Import the previous part

Run the cell below — a file picker opens; select the zip(s) you downloaded earlier:

- `nb1_mdlm_outputs.zip` — all MDLM + ablation results so the final tables include the primary rows

*(Re-running this notebook after a disconnect? Also re-upload **this** part's own output zip — finished stages are skipped and training resumes from its checkpoint.)* Using Drive? Set `USE_DRIVE=True` above and skip this.

In [ ]:
# === Import previous outputs (pick the .zip file(s) listed above) ===
if not USE_DRIVE:
    from siflow.utils.io import import_zips
    import_zips(BASE)
else:
    print('USE_DRIVE: reading prior outputs from', BASE)

In [ ]:
# === Sizes (shrink for a smoke; total stays well under one 12h session) ===
DREAM_STEPS = 8000
LLADA_STEPS = 8000
N_DREAM_TOK = 40_000
N_LLADA_TOK = 40_000
print("sizes set")

In [ ]:
# === SIFLOW-D: Dream-org/Dream-v0-Base-7B (head-only, guarded + resumable) ===
import os
from transformers import AutoTokenizer
from siflow.data import build_token_chunks
ip = get_ipython()
if not os.path.exists(f"{BASE}/results/dream.json"):
    tk = AutoTokenizer.from_pretrained("Dream-org/Dream-v0-Base-7B", trust_remote_code=True)
    if not os.path.exists(f"{BASE}/data/dream_256.npy"):
        build_token_chunks(tk, 256, N_DREAM_TOK, f"{BASE}/data/dream_256.npy",
                           dataset="Skylion007/openwebtext", split="train")
    if not os.path.exists(f"{BASE}/data/dream_val.npy"):
        build_token_chunks(tk, 256, 1_500, f"{BASE}/data/dream_val.npy",
                           dataset="Skylion007/openwebtext", split="train", skip_seqs=N_DREAM_TOK)
    del tk
    ip.system(f"python scripts/train.py --config siflow/config/dream.yaml --set "
              f"data.tokens_path={BASE}/data/dream_256.npy out_dir={BASE}/ckpt/dream "
              f"run_id=siflow_dream train.total_steps={DREAM_STEPS}")
    ip.system(f"python scripts/evaluate.py --config siflow/config/dream.yaml --system siflow --variant D "
              f"--ckpt-dir {BASE}/ckpt/dream --ref-tokens {BASE}/data/dream_val.npy "
              f"--n-samples 500 --k-list 1 4 --straightness-n 0 --out {BASE}/results/dream.json")
    ip.system(f"python scripts/evaluate.py --config siflow/config/dream.yaml --system teacher --variant D "
              f"--steps 64 --ref-tokens {BASE}/data/dream_val.npy --n-samples 300 "
              f"--out {BASE}/results/dream_teacher.json")
else:
    print("SIFLOW-D already done; skipping")

In [ ]:
# === SIFLOW-L: GSAI-ML/LLaDA-8B-Base (head-only, guarded + resumable) ===
import os
from transformers import AutoTokenizer
from siflow.data import build_token_chunks
ip = get_ipython()
if not os.path.exists(f"{BASE}/results/llada.json"):
    tk = AutoTokenizer.from_pretrained("GSAI-ML/LLaDA-8B-Base", trust_remote_code=True)
    if not os.path.exists(f"{BASE}/data/llada_256.npy"):
        build_token_chunks(tk, 256, N_LLADA_TOK, f"{BASE}/data/llada_256.npy",
                           dataset="Skylion007/openwebtext", split="train")
    if not os.path.exists(f"{BASE}/data/llada_val.npy"):
        build_token_chunks(tk, 256, 1_500, f"{BASE}/data/llada_val.npy",
                           dataset="Skylion007/openwebtext", split="train", skip_seqs=N_LLADA_TOK)
    del tk
    ip.system(f"python scripts/train.py --config siflow/config/llada.yaml --set "
              f"data.tokens_path={BASE}/data/llada_256.npy out_dir={BASE}/ckpt/llada "
              f"run_id=siflow_llada train.total_steps={LLADA_STEPS}")
    ip.system(f"python scripts/evaluate.py --config siflow/config/llada.yaml --system siflow --variant L "
              f"--ckpt-dir {BASE}/ckpt/llada --ref-tokens {BASE}/data/llada_val.npy "
              f"--n-samples 500 --k-list 1 4 --straightness-n 0 --no-mauve --out {BASE}/results/llada.json")
    ip.system(f"python scripts/evaluate.py --config siflow/config/llada.yaml --system teacher --variant L "
              f"--steps 64 --ref-tokens {BASE}/data/llada_val.npy --n-samples 300 --no-mauve "
              f"--out {BASE}/results/llada_teacher.json")
else:
    print("SIFLOW-L already done; skipping")

In [ ]:
# === FINAL combined tables + figures (MDLM + Dream-7B + LLaDA-8B) ===
ip = get_ipython()
ip.system(f"python scripts/make_tables.py  --results {BASE}/results --out {BASE}/tables_auto.tex")
ip.system(f"python scripts/make_figures.py --results {BASE}/results --out-dir {BASE}/figures")
print(open(f"{BASE}/tables_auto.tex").read())

In [ ]:
# === Save + auto-download this part's output ===
from siflow.utils.io import export_and_download
if not USE_DRIVE:
    export_and_download(BASE, 'nb2_final_paper_artifacts.zip', include=['results', 'figures', 'tables_auto.tex', 'ckpt/dream', 'ckpt/llada'])
else:
    print('USE_DRIVE: outputs already persisted under', BASE)

**Done.** Unzip `nb2_final_paper_artifacts.zip` into `paper/` (drop `tables_auto.tex` and `figures/*.pdf` in) and recompile `paper/siflow_aaai.tex` — Tables 2–3 and the figures populate with MDLM, Dream-7B (-D) and LLaDA-8B (-L) rows.